# Train YOLO detect board tren Google Colab (dataset tu Roboflow)

Pipeline:
1. LOCAL: chup anh (`capture_dataset.py`) -> auto nhan HSV (`auto_label_hsv.py`).
2. ROBOFLOW: upload `dataset/images` + `dataset/labels` (nhan HSV lam san) -> sua box -> Generate -> Export (YOLOv11) -> lay snippet API.
3. COLAB: chay notebook nay (bat GPU), dan snippet Roboflow -> train -> tai `best.pt`.

**Nho bat GPU:** Runtime -> Change runtime type -> T4 GPU.

## 1. Kiem tra GPU + cai dat

In [ ]:
!nvidia-smi
!pip install -q ultralytics roboflow

## 2. Tai dataset tu Roboflow

Tren Roboflow: Export -> format **YOLOv11** -> **show download code** -> copy snippet vao day.
Doi `api_key`, `workspace`, `project`, `version` cho dung project cua ban.

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_API_KEY")
project = rf.workspace("YOUR_WORKSPACE").project("YOUR_PROJECT")
dataset = project.version(1).download("yolov11")

DATA_YAML = dataset.location + "/data.yaml"
print("data.yaml:", DATA_YAML)
print(open(DATA_YAML).read())

## 3. Train

Board to, 1 class -> `yolo11n` da du. Muon chinh xac hon doi sang `yolo11s`/`yolo11m`.
Neu da bat augmentation tren Roboflow thi co the giam bot augment o day.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
model.train(
    data=DATA_YAML,
    epochs=100, imgsz=960, batch=16, patience=30,
    name='board_yolo', project='/content/runs',
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=180, translate=0.1, scale=0.5, fliplr=0.5, flipud=0.5, mosaic=1.0,
)

## 4. Xem ket qua train

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('/content/runs/board_yolo/*.png'))[:4]:
    print(p); display(Image(filename=p, width=600))

## 5. Tai model ve

Tai `best.pt` ve -> dat vao local: `yolo_board/runs/board_yolo/weights/best.pt`
-> dung duoc ngay voi `detect_board.py`.

In [ ]:
from google.colab import files
files.download('/content/runs/board_yolo/weights/best.pt')

## (Tuy chon) Luu vao Google Drive de khoi mat khi het phien

```python
from google.colab import drive
drive.mount('/content/drive')
import shutil
shutil.copy('/content/runs/board_yolo/weights/best.pt', '/content/drive/MyDrive/board_best.pt')
```